# Analise de negocio

Este notebook responde as duas teses de negocio definidas para este MVP, substituindo as
perguntas genericas da versao anterior (atraso->satisfacao simples; vendas por estado):

1. **Os custos de frete estao adequados?** Quais categorias pagam mais frete por quilo/volume
   transportado, e isso e proporcional ao porte fisico do produto ou indica ineficiencia
   logistica/pricing?
2. **Os vendedores tem respeitado os prazos de entrega?** Um grupo pequeno de vendedores
   concentra a maior parte dos atrasos e notas baixas (padrao 80/20), ou o problema esta
   distribuido por todos?

Fontes: `${catalog}.olist_gold.fato_vendas` e `${catalog}.olist_gold.dim_produtos`. Cada consulta
e seguida de uma celula markdown de interpretacao, a ser preenchida com os valores reais obtidos
na execucao no Databricks.


In [0]:
-- Pergunta 1: custo de frete relativo ao porte fisico, por categoria de produto.
WITH item_metrics AS (
  SELECT
    f.order_id,
    f.product_id,
    f.freight_value,
    p.product_category_name,
    p.product_weight_g,
    (p.product_length_cm * p.product_height_cm * p.product_width_cm) / 1000000.0 AS volume_m3
  FROM ${catalog}.olist_gold.fato_vendas f
  JOIN ${catalog}.olist_gold.dim_produtos p USING (product_id)
  WHERE p.product_weight_g IS NOT NULL
    AND p.product_length_cm IS NOT NULL
    AND p.product_height_cm IS NOT NULL
    AND p.product_width_cm IS NOT NULL
),
categoria_agg AS (
  SELECT
    product_category_name,
    COUNT(*) AS total_itens,
    ROUND(AVG(freight_value), 2) AS frete_medio,
    ROUND(AVG(product_weight_g), 0) AS peso_medio_g,
    ROUND(AVG(volume_m3), 4) AS volume_medio_m3,
    ROUND(AVG(freight_value) / NULLIF(AVG(product_weight_g) / 1000.0, 0), 2) AS frete_medio_por_kg
  FROM item_metrics
  GROUP BY product_category_name
  HAVING COUNT(*) >= 30
)
SELECT * FROM categoria_agg ORDER BY frete_medio_por_kg DESC;


In [0]:
%python
# Visualizacao: frete medio por kg, por categoria (ordenado), destacando as pontas mais caras/baratas.
import matplotlib.pyplot as plt

pdf_frete = _sqldf.toPandas().sort_values("frete_medio_por_kg", ascending=True)

limite_caro = pdf_frete["frete_medio_por_kg"].quantile(0.85)
limite_barato = pdf_frete["frete_medio_por_kg"].quantile(0.15)
cores = [
    "#d62728" if v >= limite_caro else "#2ca02c" if v <= limite_barato else "#7f7f7f"
    for v in pdf_frete["frete_medio_por_kg"]
]

plt.figure(figsize=(10, 14))
plt.barh(pdf_frete["product_category_name"], pdf_frete["frete_medio_por_kg"], color=cores)
plt.xlabel("Frete medio por kg (R$)")
plt.ylabel("Categoria de produto")
plt.title(
    "Frete medio por kg, por categoria\n"
    "vermelho = mais caro por kg (itens leves/pequenos) | verde = mais barato por kg (itens pesados/volumosos)"
)
plt.tight_layout()
plt.show()


### Conclusao da Pergunta 1: Os custos de frete estao adequados?
**Nao.** O custo de frete e muito alto para produtos pequenos/leves quando comparado a produtos
grandes/pesados. As 3 categorias com **pior** `frete_medio_por_kg` sao `telefonia` (R$ 59,89/kg;
peso medio 262 g, volume medio 0,0018 m³), `fashion_esporte` (R$ 56,68/kg; 340 g) e
`fashion_underwear_e_moda_praia` (R$ 52,97/kg; 276 g) — todas categorias de itens leves e de
volume pequeno. As 3 **melhores** sao `moveis_escritorio` (R$ 3,56/kg; peso medio 11,4 kg),
`moveis_quarto` (R$ 4,25/kg; 10,0 kg) e `moveis_sala` (R$ 4,41/kg; 8,1 kg) — moveis pesados e
volumosos, ate 17x mais baratos por kg.

O padrao **nao e proporcional ao porte fisico**: entre essas pontas, o frete absoluto medio
cresce bem menos (de R$ 15,67 para R$ 40,55, ~2,6x) do que o peso (de 262 g para 11.390 g,
~43x). Isso indica que o custo de frete tem um piso/tarifa minima por envio que varia pouco com
peso ou volume, penalizando desproporcionalmente itens leves (telefonia, moda) e diluindo o
custo nos itens pesados (moveis). Recomendacao: verificar alternativas para baratear o custo de
frete de produtos pequenos/leves, revisando a tarifa minima ou a estrutura de frete fixo para
pedidos leves.


In [0]:
-- Pergunta 2: concentracao de atrasos e notas baixas por vendedor.
WITH pedidos_vendedor AS (
  SELECT
    f.seller_id,
    f.order_id,
    f.delivery_status,
    MAX(CAST(f.review_score AS INT)) AS review_score
  FROM ${catalog}.olist_gold.fato_vendas f
  GROUP BY f.seller_id, f.order_id, f.delivery_status
),
seller_agg AS (
  SELECT
    seller_id,
    COUNT(*) AS total_pedidos,
    SUM(CASE WHEN delivery_status IN ('atraso_1_7_dias', 'atraso_mais_de_7_dias') THEN 1 ELSE 0 END) AS pedidos_atrasados,
    SUM(CASE WHEN review_score IN (1, 2) THEN 1 ELSE 0 END) AS pedidos_nota_baixa
  FROM pedidos_vendedor
  GROUP BY seller_id
  HAVING COUNT(*) >= 10
),
ranked AS (
  SELECT *, NTILE(10) OVER (ORDER BY pedidos_atrasados DESC, pedidos_nota_baixa DESC) AS decil_risco
  FROM seller_agg
)
SELECT
  decil_risco,
  COUNT(*) AS qtd_vendedores,
  SUM(total_pedidos) AS total_pedidos,
  SUM(pedidos_atrasados) AS total_atrasados,
  SUM(pedidos_nota_baixa) AS total_nota_baixa,
  ROUND(100.0 * SUM(pedidos_atrasados) / SUM(SUM(pedidos_atrasados)) OVER (), 2) AS pct_do_total_atrasos,
  ROUND(100.0 * SUM(pedidos_nota_baixa) / SUM(SUM(pedidos_nota_baixa)) OVER (), 2) AS pct_do_total_notas_baixas
FROM ranked
GROUP BY decil_risco
ORDER BY decil_risco;


In [0]:
%python
# Visualizacao: concentracao de atrasos e notas baixas por decil de risco de vendedor.
import numpy as np
import matplotlib.pyplot as plt

pdf_decis = _sqldf.toPandas().sort_values("decil_risco")

x = np.arange(len(pdf_decis["decil_risco"]))
largura = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - largura / 2, pdf_decis["pct_do_total_atrasos"], largura, label="% do total de atrasos", color="#d62728")
ax.bar(x + largura / 2, pdf_decis["pct_do_total_notas_baixas"], largura, label="% do total de notas baixas (1-2)", color="#ff7f0e")

ax.axhline(10, color="gray", linestyle="--", linewidth=1, label="10% esperado (distribuicao uniforme)")
ax.set_xlabel("Decil de risco do vendedor (1 = maior risco, 10 = menor risco)")
ax.set_ylabel("% do total")
ax.set_title("Concentracao de atrasos e notas baixas por decil de risco de vendedor")
ax.set_xticks(x)
ax.set_xticklabels(pdf_decis["decil_risco"])
ax.legend()
plt.tight_layout()
plt.show()


### Conclusao da Pergunta 2: Os vendedores tem respeitado os prazos de entrega?
**Nem sempre, e de forma bastante concentrada.** O `decil_risco = 1` (128 vendedores, ~10% dos
1.271 vendedores com pelo menos 10 pedidos) concentra **58,97%** de todos os pedidos atrasados e
**53,62%** de todas as notas baixas (1-2) da base — bem acima dos 10% esperados caso o problema
estivesse distribuido uniformemente. Somando os decis 1+2 (**20% dos vendedores**), a
concentracao sobe para **quase 75%** dos atrasos e **68,5%** das notas baixas, enquanto os dois
melhores decis (os 20% melhores vendedores) **nao registraram nenhum atraso**.

Isso **confirma o padrao 80/20**: o problema de atraso e insatisfacao esta fortemente
concentrado em um grupo pequeno de vendedores, e nao distribuido pela base. Recomendacoes:
priorizar maior auditoria e um plano de acao comercial/logistico focado nesses ~128 piores
vendedores do decil 1 (renegociacao de SLA, revisao de transportadora, ou desligamento em casos
extremos). Como contrapartida, oferecer premiacoes aos melhores vendedores — maior visibilidade
de seus produtos, um selo de qualidade, entre outros incentivos.


## Resposta executiva

- **Os custos de frete estao adequados?** Nao. O custo e muito alto para produtos pequenos/leves
  quando comparado a produtos grandes/pesados. Categorias leves e de pouco volume (`telefonia`,
  `fashion_esporte`, `fashion_underwear_e_moda_praia`) pagam entre R$ 53 e R$ 60 por kg, enquanto
  categorias de moveis pesados (`moveis_escritorio`, `moveis_quarto`, `moveis_sala`) pagam entre
  R$ 3,56 e R$ 4,41 por kg — ate 17x menos. O frete absoluto cresce muito pouco entre essas
  pontas (~2,6x) frente ao peso (~43x), o que aponta para uma tarifa minima/fixa por envio que
  penaliza itens leves. Recomendacao: verificar alternativas para baratear o custo de frete para
  produtos pequenos/leves, em vez de assumir que o custo atual reflete corretamente o porte
  fisico.

- **Os vendedores tem respeitado os prazos de entrega?** Nem sempre, e de forma concentrada.
  Cerca de **20% dos vendedores** (decis 1+2, ~256 vendedores) concentram **quase 75%** dos
  atrasos e **68,5%** das notas baixas da base, enquanto os melhores 20% nao registraram nenhum
  atraso. Recomendacao: priorizar maior auditoria e um plano de acao comercial focado nesses
  piores vendedores (renegociacao de SLA, revisao de transportadora, ou desligamento em casos
  extremos). Outra sugestao e oferecer premiacoes aos melhores vendedores, seja com maior
  visibilidade de seus produtos, um selo de qualidade, etc.
